# APAC Risk Monitor — Week 2 risk engine

Historical VaR/ES (loss-positive), Euler risk contribution, and first-order stress scenarios.
All parameters come from `config/risk.yaml`. Requires Week 1 processed returns (`python run_week1.py`).

Run from the repository root (or with that directory on `PYTHONPATH`).

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipeline import run_week2_pipeline
from src.scenarios import APPROXIMATION_DISCLAIMER

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
result = run_week2_pipeline()
print(result.portfolio_config.portfolio_name)
print("Loss convention: VaR/ES are positive numbers representing losses.")
result.metrics.summary

## Rolling volatility and drawdown

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
result.metrics.rolling_vol.plot(ax=axes[0], color="C0", linewidth=1.2)
axes[0].set_ylabel("Ann. vol")
axes[0].set_title(f"Rolling volatility ({result.risk_config.performance.rolling_vol_window}d)")
axes[0].grid(True, alpha=0.3)
result.metrics.drawdown_series.plot(ax=axes[1], color="C3", linewidth=1.2)
axes[1].set_ylabel("Drawdown")
axes[1].set_title("Drawdown from peak")
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Asset correlation heatmap

In [ ]:
corr = result.metrics.correlation
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.index)), corr.index)
ax.set_title("Asset return correlation")
fig.colorbar(im, ax=ax, fraction=0.046)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.tight_layout()
plt.show()
corr

## VaR and Expected Shortfall

Loss-positive: a value of 0.012 is a **1.2% loss**. Historical is the configured method; parametric (Gaussian) is a comparison only.

In [ ]:
var_rows = [
    {
        "confidence": est.confidence,
        "lookback_used": est.lookback_used,
        "horizon_days": est.horizon_days,
        "historical_var": est.historical_var,
        "historical_es": est.historical_es,
        "parametric_var": est.parametric_var,
        "parametric_es": est.parametric_es,
    }
    for est in result.metrics.var_es
]
pd.DataFrame(var_rows)

## Risk contribution (Euler)

`outsized_risk_flag` is true when contribution share / weight share exceeds the ratio in `config/risk.yaml`.

In [ ]:
contrib = result.contribution.table.copy()
print(f"Portfolio vol (ann.): {result.contribution.portfolio_vol:.4%}")
contrib[contrib["level"] == "position"][
    ["name", "sleeve", "weight", "standalone_vol", "vol_contribution", "share_ratio", "outsized_risk_flag"]
]

## Scenario summary (historical + hypothetical + combined)

Every row carries: *First-order approximation — ignores convexity, correlation shifts, and second-round effects*

In [ ]:
print(APPROXIMATION_DISCLAIMER)
for w in result.scenarios.warnings:
    print("WARN:", w)
result.scenarios.table[["scenario", "type", "pnl_pct", "pnl_currency", "worst_hit_sleeve", "disclaimer"]]

## Key observations

Fill in after reviewing the tables and charts:

- Observation 1: _(what dominates standalone vs contribution-to-vol?)_
- Observation 2: _(how does historical VaR compare with parametric at 95% / 99%?)_
- Observation 3: _(which historical window or hypothetical shock is the binding stress?)_
- Observation 4: _(any "small position, outsized risk" flags — do they match the sleeve story?)_
- Observation 5: _(what would you change in `config/risk.yaml` before treating these numbers as decision-useful?)_